# Avito: кандидатогенерация — эмбеддинги (подготовка v4)

Ноутбук готовит **генератор кандидатов по смыслу**: дообучает двухбашенный энкодер на парах «запрос — выбранное объявление» из train и кодирует им корпус и все запросы, нужные дальше. Результат — артефакт, который использует `04_ranker_embeddings.ipynb`.

**Зачем.** Две отправки показали, что улучшения ранжирования на лидерборд не переносятся (v3: +0,037 на валидации, +0,001 на лидерборде), а небольшой прирост дало расширение пула кандидатов. У 62% запросов бенчмарка текст не встречался в train, и лексический поиск их не находит: «перевозка груза» → «грузоперевозки», «замена гидроаккумулятора» → «ремонт скважин». Значит, вкладываться нужно в полноту пула.

| Раздел | Что происходит | Время на A100 (20 ГБ) |
|---|---|---|
| 0. Окружение | настройки, код, пакеты, GPU и память | 1 мин |
| 1. Данные и выборки | train, бенчмарк, выборки v4, тексты | 3–5 мин |
| 2. Выбор модели | Recall@100 двух моделей без дообучения | ~10 мин |
| 3. Дообучение | подбор батча, трудные негативы, обучение | 40–60 мин |
| 4. Результат | Recall@100 после дообучения | ~5 мин |
| 5. Артефакт | вектора объявлений и запросов, модель | ~5 мин |

**Честность оценки.** Запросы, которые в `04` попадут в валидацию и в фолды ранкера, исключены из обучения энкодера: иначе он их запомнит и метрика будет завышена. Выборки строятся теми же функциями (`src/sampling.py`), что и в `04`.

**Воспроизводимость.** Обучение на GPU повторяется бит-в-бит только на том же железе, поэтому результат этого ноутбука — артефакт (манифест с md5 всех файлов), а ноутбук `04` по нему считает ответ на CPU детерминированно.

**Как запускать:** см. `JUPYTERLAB.md` в корне репозитория. Коротко: сначала `DRY_RUN = True` (~10 минут), затем полный прогон; долгий прогон лучше запускать из терминала через papermill, чтобы он не зависел от вкладки браузера.

## Настройки запуска

Единственная ячейка, которую может понадобиться поправить. Значение `None` означает «определить автоматически». Эта же ячейка помечена тегом `parameters`, поэтому при запуске через papermill любую настройку можно передать снаружи: `-p DRY_RUN True`.

In [ ]:
# ── Пути (None — автоматически) ──────────────────────────────────────────────────────────
REPO_DIR = None       # папка репозитория, если ноутбук открыт не из его папки notebooks/
DATA_DIR = None       # папка с train.parquet и benchmark_*.parquet; по умолчанию <репозиторий>/data
WORK_DIR = None       # куда сохранить артефакт; по умолчанию <репозиторий>/artifacts
HF_HOME = None        # кэш моделей Hugging Face (~1,1 ГБ на модель); по умолчанию ~/.cache/huggingface
HF_ENDPOINT = None    # зеркало, если huggingface.co недоступен, например "https://hf-mirror.com"

# ── Режим ─────────────────────────────────────────────────────────────────────────────────
DRY_RUN = False       # True — все шаги на маленьких подвыборках (~10 минут): проверка перед полным запуском
SMOKE_TEST = False    # True — крошечная модель со случайными весами, ничего не скачивается (проверка кода)

# ── Обучение (None / 0 — значения из src/config.py) ──────────────────────────────────────
MODEL_CANDIDATES = None   # список имён на Hugging Face или путей к скачанным папкам;
                          # ["intfloat/multilingual-e5-base"] — без сравнения моделей (экономит ~15 минут)
BATCH_SIZE = 0            # 0 — подобрать под память GPU
TRAIN_PAIRS = None        # пар для дообучения; None — 400 000

## 0. Окружение

Ячейка переносит настройки в переменные окружения, находит репозиторий (поднимаясь от текущей папки), подключает его код и ставит **только недостающие** лёгкие пакеты нужных версий. Уже установленные numpy, pandas и torch не трогаются: переустановка torch в готовой GPU-среде может сломать поддержку CUDA. Число потоков BLAS фиксируется **до** импорта numpy — это часть воспроизводимости.

In [ ]:
import base64, importlib, os, subprocess, sys
from pathlib import Path

# 1. Настройки → переменные окружения (их читают модули src). Уже заданные снаружи не сбрасываются.
for _name in ("REPO_DIR", "DATA_DIR", "WORK_DIR", "OUTPUT_DIR", "EMB_DIR", "HF_HOME", "HF_ENDPOINT"):
    if globals().get(_name):
        os.environ[_name] = str(globals()[_name])
for _name in ("DRY_RUN", "SMOKE_TEST"):
    if globals().get(_name):
        os.environ[_name] = "1"

# 2. Фиксированное число потоков BLAS — до импорта numpy
N_THREADS = 4
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = str(N_THREADS)

# 3. Код решения
REPO_URL = "https://github.com/mishin-mikhail/avito_autumn_dev.git"
REPO_REF = "main"            # ветка, тег или хеш коммита — используется, только если репозиторий клонирует сам ноутбук


def _github_token():
    """GITHUB_TOKEN из окружения или из Kaggle Secrets (None, если его нет)."""
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


def _git(*args, token=None) -> str:
    """git без утечки токена: заголовок авторизации передаётся через переменные окружения."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git завершился с ошибкой:\n{error}")
    return result.stdout.strip()


def find_repo_root() -> Path:
    """REPO_DIR → папки выше текущей → (Kaggle или GITHUB_TOKEN) клонирование в текущую папку."""
    candidates = [Path(os.environ["REPO_DIR"]).expanduser()] if os.environ.get("REPO_DIR") else []
    candidates += [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / "src" / "pipeline.py").exists():
            return path.resolve()
    token = _github_token()
    if not (token or Path("/kaggle/input").exists()):
        raise RuntimeError(
            "Не найден код решения (папка src/). Откройте ноутбук из папки notebooks/ клонированного "
            "репозитория или укажите путь к репозиторию в настройке REPO_DIR.")
    target = Path("/kaggle/working/avito-candgen") if Path("/kaggle/working").exists() else Path.cwd() / "avito-candgen"
    if not target.exists():
        _git("clone", "--quiet", REPO_URL, str(target), token=token)
    _git("-C", str(target), "fetch", "--quiet", "--tags", "--force", "origin", token=token)
    is_branch = subprocess.run(["git", "-C", str(target), "rev-parse", "--verify", "--quiet",
                                f"origin/{REPO_REF}"], capture_output=True).returncode == 0
    _git("-C", str(target), "checkout", "--quiet", "--force", "--detach",
         f"origin/{REPO_REF}" if is_branch else REPO_REF)
    return target


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
try:
    COMMIT = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip() or "(не git-репозиторий)"
except FileNotFoundError:
    COMMIT = "(git не установлен)"


# 4. Недостающие пакеты. Ставятся в окружение текущего ядра; при нехватке прав — в --user.
def ensure_packages(requirements: dict) -> None:
    """requirements: модуль → pip-спецификации. Ставит только то, чего нет."""
    missing = []
    for module, specs in requirements.items():
        try:
            importlib.import_module(module)
        except ImportError:
            missing += specs
    if not missing:
        return
    print("устанавливаю:", " ".join(missing))
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    if subprocess.run(cmd).returncode != 0 and subprocess.run(cmd + ["--user"]).returncode != 0:
        raise RuntimeError(f"Не удалось установить {missing}. Установите их вручную в терминале.")
    importlib.invalidate_caches()
    import site
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())


ensure_packages({
    "pyarrow": ["pyarrow"],
    "pymorphy3": ["pymorphy3==2.0.6", "pymorphy3-dicts-ru==2.4.417150.4580142"],
    "transformers": ["transformers==4.57.6"],
})
try:
    import torch
except ImportError as error:
    raise RuntimeError("В окружении нет PyTorch. Ставить его нужно под версию CUDA этой машины — "
                       "команда есть на https://pytorch.org/get-started/locally/") from error
print(f"репозиторий: {REPO_ROOT}\nкоммит: {COMMIT}")

In [ ]:
import json
import time
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import display

from src import encoder as enc
from src.config import CFG, EMB_CFG, RANKER_CFG
from src.data import load_benchmark, load_train
from src.paths import get_data_dir, get_work_dir
from src.pipeline import add_lemma_keys
from src.repro import library_versions, seed_everything
from src.sampling import build_folds, build_validation, group_table, picked_rows_mask, scheme_keys
from src.text import Lemmatizer
from src.utils import resources_report, timer
from src.validation import add_query_segments, mark_seen

seed_everything(CFG.seed)
SMOKE = os.environ.get("SMOKE_TEST") == "1"
DRY = os.environ.get("DRY_RUN") == "1" or SMOKE
START = time.perf_counter()

# Конфиги: значения из src/config.py + настройки из первой ячейки
E = replace(EMB_CFG, candidates=tuple(MODEL_CANDIDATES or EMB_CFG.candidates),
            batch_size=BATCH_SIZE or EMB_CFG.batch_size,
            train_pairs=EMB_CFG.train_pairs if TRAIN_PAIRS is None else TRAIN_PAIRS)
R = replace(RANKER_CFG, version="v4")
if DRY:           # быстрый прогон всех шагов на подвыборках; выборки — как в SMOKE-режиме ноутбука 04
    E = replace(E, train_pairs=2_000, zero_shot_queries=200, encode_batch=64)   # подбор батча — как в полном
    R = replace(R, n_val_queries=300, fold_queries=300)

DATA_DIR, WORK_DIR = get_data_dir(), get_work_dir()
EMB_DIR = WORK_DIR / "embeddings"
GPU = enc.device_info()
AMP = enc.choose_amp(E.amp_dtype, GPU)

print(f"режим: {'SMOKE_TEST' if SMOKE else 'DRY_RUN' if DRY else 'полный прогон'}")
print(f"данные: {DATA_DIR}\nартефакт: {EMB_DIR}")
print(f"устройство: {GPU['name']}" + (f", {GPU['memory_gb']} ГБ, вычисления в {AMP.name}" if GPU["device"] == "cuda" else ""))
resources_report(WORK_DIR, need_ram_gb=10, need_disk_gb=5)
print(library_versions())
if GPU["device"] != "cuda" and not DRY:
    raise RuntimeError("GPU не найден: полный прогон на CPU займёт сутки. Проверьте, что ядро запущено "
                       "в GPU-окружении (nvidia-smi в терминале), или включите DRY_RUN для проверки кода.")

## 1. Данные и выборки

Строим те же выборки валидации и фолдов, что и в `04`, — для **обеих** схем валидации (`injected` и `in_corpus`): какая из них будет выбрана, решается только в `04`. Из обучения энкодера исключаются строки самих выбранных запросов: у «новых» — все строки их текста, у «знакомых» — строки их группы.

In [ ]:
with timer("загрузка"):
    # описания нужны для текстов объявлений, поэтому train грузится целиком (~3,5 ГБ в памяти)
    train = load_train(DATA_DIR, with_description=True)
    bench_q, bench_items = load_benchmark(DATA_DIR)

lem = Lemmatizer()
with timer("подготовка запросов"):
    add_lemma_keys(train, lem)
    add_lemma_keys(bench_q, lem)
    item_locations = pd.Index(pd.concat([train["item_location_id"], bench_items["item_location_id"]]).unique())
    add_query_segments(train, item_locations)
    add_query_segments(bench_q, item_locations)
    mark_seen(bench_q, train["norm_text"].unique())
    groups = group_table(train, bench_items["item_id"])

with timer("выборки v4"):
    VAL = build_validation(train, groups, bench_q, R)
    KEYS = scheme_keys(groups)
    FOLDS = {name: build_folds(train, groups, bench_q, R, VAL[name], KEYS[name]) for name in VAL}
    SAMPLES = [*VAL.values(), *(fold for folds in FOLDS.values() for fold in folds)]

EXCLUDED = picked_rows_mask(train, SAMPLES)
print(f"выборок: {len(SAMPLES)} ({sum(len(s.queries) for s in SAMPLES):,} запросов)")
print(f"строк train: {len(train):,}; исключено из обучения энкодера: {EXCLUDED.sum():,} ({EXCLUDED.mean():.3f})")

In [ ]:
# Что кодировать для ноутбука 04:
#  * объявления — корпус бенчмарка + эталоны выборок, которых в нём нет (04 подмешивает их в корпус валидации);
#  * запросы — бенчмарк и все запросы выборок (04 не запускает нейросеть и берёт вектора отсюда).
bench_ids = frozenset(bench_items["item_id"])
extra_ids = sorted(dict.fromkeys(i for s in SAMPLES for rel in s.truth for i in rel if i not in bench_ids))
extra_items = (train[train["item_id"].isin(pd.Index(extra_ids))]
               .drop_duplicates("item_id", keep="first")[bench_items.columns].reset_index(drop=True))
QUERY_TEXTS_04 = list(dict.fromkeys(enc.query_texts(bench_q) + [t for s in SAMPLES for t in enc.query_texts(s.queries)]))

corpus_rows = np.arange(len(bench_items))
if DRY:           # в быстром режиме кодируем только часть корпуса
    corpus_rows = corpus_rows[:5000]
CORPUS_ITEMS = pd.concat([bench_items.iloc[corpus_rows], extra_items], ignore_index=True)
CORPUS_TEXTS = enc.item_texts(CORPUS_ITEMS, E.desc_chars)
corpus_row = {v: i for i, v in enumerate(CORPUS_ITEMS["item_id"])}

# Оценка поиска по векторам: пары выбранных запросов (энкодер их не видит), чьё объявление есть в корпусе
eval_rows = train[EXCLUDED & train["item_id"].isin(pd.Index(CORPUS_ITEMS["item_id"])).to_numpy()]
eval_rows = eval_rows.drop_duplicates("query_key").head(E.zero_shot_queries)
EVAL_QUERIES = enc.query_texts(eval_rows)
EVAL_POSITIVE = np.array([corpus_row[i] for i in eval_rows["item_id"]])

# Пары для обучения — детерминированная подвыборка строк, не относящихся к выборкам
free = np.flatnonzero(~EXCLUDED)
if E.train_pairs and len(free) > E.train_pairs:
    free = np.sort(np.random.default_rng(CFG.seed).choice(free, E.train_pairs, replace=False))
pairs_df = train.iloc[free]

print(f"объявлений к кодированию: {len(CORPUS_TEXTS):,} (корпус {len(corpus_rows):,} + подмешиваемые {len(extra_items):,})")
print(f"запросов для 04: {len(QUERY_TEXTS_04):,} | оценочных пар: {len(EVAL_QUERIES):,} | пар для обучения: {len(pairs_df):,}")
print("пример запроса:   ", EVAL_QUERIES[0][:150])
print("пример объявления:", CORPUS_TEXTS[0][:150])

## 2. Выбор модели

Обе кандидатки — из семейства e5 (префиксы `query:` и `passage:`); `deepvk/USER-base` — тот же e5, дообученный на русском. Сравниваем без дообучения по Recall@100: доля запросов, у которых нужное объявление попало в топ-100 по близости векторов. Веса скачиваются здесь же: если Hugging Face недоступен, ячейка сразу скажет, что делать.

In [ ]:
def evaluate(model, tag: str):
    """Recall@100 поиска по векторам на отложенных парах. Возвращает (вектора корпуса, recall)."""
    with timer(f"{tag}: кодирование корпуса"):
        item_emb = model.encode(CORPUS_TEXTS, E.max_len_item, E.encode_batch, log_every=50)
    q_emb = model.encode(EVAL_QUERIES, E.max_len_query, E.encode_batch)
    recall = enc.dense_recall(q_emb, item_emb, EVAL_POSITIVE, k=100, device=GPU["device"])
    print(f"{tag}: Recall@100 = {recall:.4f}")
    return item_emb, recall


ZERO_SHOT, SOURCES = {}, {}
if SMOKE:
    MODEL_NAME = "debug-random"
    model = enc.build_debug_encoder(CORPUS_TEXTS + EVAL_QUERIES, GPU["device"])
    ZERO_SHOT[MODEL_NAME] = evaluate(model, MODEL_NAME)[1]
else:
    for name in E.candidates:
        with timer(f"{name}: скачивание"):
            SOURCES[name] = enc.fetch_model(name)
    best = None
    for name in E.candidates:
        candidate = enc.BiEncoder.from_pretrained(SOURCES[name], GPU["device"], AMP)
        _, ZERO_SHOT[name] = evaluate(candidate, name)
        if best is None or ZERO_SHOT[name] > ZERO_SHOT[best]:
            best, model = name, candidate
        else:
            del candidate
    MODEL_NAME = best
    if GPU["device"] == "cuda":
        torch.cuda.empty_cache()
print(f"\n→ дообучаем: {MODEL_NAME}")

## 3. Дообучение

**Размер батча.** В контрастном обучении каждый запрос батча отталкивается от объявлений всех остальных запросов, поэтому больший батч — больше негативов и лучше поиск. Батч подбирается пробным шагом на худшем случае (тексты максимальной длины, память под оптимизатор уже занята).

**Трудные негативы** — похожие объявления корпуса, которые пользователь не выбрал. Берутся из окна рангов, а не с самого верха: ближайшие объявления часто тоже подходят запросу, и отталкивать их — значит портить полноту.

In [ ]:
if E.batch_size == 0:
    with timer("подбор батча"):
        E = replace(E, batch_size=enc.probe_batch_size(model, E, candidates=tuple(
            b for b in (512, 384, 320, 256, 192, 160, 128, 96, 64, 48, 32, 16) if b <= E.max_batch_size)))
steps = len(pairs_df) // E.batch_size * E.epochs
print(f"батч: {E.batch_size} запросов → {E.batch_size * (2 + E.hard_negatives)} текстов на шаг, "
      f"негативов на запрос: {E.batch_size * (1 + E.hard_negatives) - 1}; шагов обучения: {steps:,}")

In [ ]:
TRAIN_QUERIES = enc.query_texts(pairs_df)
TRAIN_ITEM_TEXTS = enc.item_texts(pairs_df, E.desc_chars)

with timer("кодирование для поиска негативов"):
    corpus_emb = model.encode(CORPUS_TEXTS, E.max_len_item, E.encode_batch, log_every=100)
    train_q_emb = model.encode(TRAIN_QUERIES, E.max_len_query, E.encode_batch, log_every=100)

positive_in_corpus = np.array([corpus_row.get(i, -1) for i in pairs_df["item_id"]])
with timer("трудные негативы"):
    neg_idx = enc.mine_hard_negatives(train_q_emb, corpus_emb, positive_in_corpus, E.hard_negatives,
                                      E.hard_neg_skip, E.hard_neg_depth, CFG.seed, GPU["device"])
negatives = [[CORPUS_TEXTS[j] for j in row] for row in neg_idx]
pairs = enc.TrainPairs(queries=TRAIN_QUERIES, positives=TRAIN_ITEM_TEXTS, negatives=negatives)
del corpus_emb, train_q_emb
print("запрос:  ", TRAIN_QUERIES[0][:120])
print("позитив: ", TRAIN_ITEM_TEXTS[0][:120])
print("негатив: ", negatives[0][0][:120])

In [ ]:
with timer("дообучение"):
    model, history = enc.train_biencoder(model, pairs, E, CFG.seed, log_every=10 if DRY else 100)
model.save(EMB_DIR / "model")          # сохраняем сразу: обучение — самый долгий шаг
history.tail(5)

## 4. Что дало дообучение

Recall@100 на тех же отложенных парах, что и в разделе 2. Эти запросы в обучении не участвовали. Если дообучение не улучшило метрику, запускать `04` смысла нет — сначала разберёмся с параметрами обучения.

In [ ]:
ITEM_EMB, RECALL_TUNED = evaluate(model, f"{MODEL_NAME} (дообученная)")
result = pd.Series({**ZERO_SHOT, f"{MODEL_NAME} (дообученная)": RECALL_TUNED}, name="Recall@100")
display(result.round(4).to_frame())

## 5. Артефакт

В папку `artifacts/embeddings` пишутся модель, вектора объявлений и вектора запросов (float16) и манифест с параметрами, метриками и md5 всех файлов. Ноутбук `04` берёт вектора отсюда и нейросеть не запускает.

In [ ]:
with timer("кодирование запросов для 04"):
    QUERY_EMB = model.encode(QUERY_TEXTS_04, E.max_len_query, E.encode_batch, log_every=50)

MANIFEST = enc.save_artifact(
    EMB_DIR, CORPUS_ITEMS["item_id"], ITEM_EMB, QUERY_TEXTS_04, QUERY_EMB,
    meta={"model_name": MODEL_NAME, "model_source": str(SOURCES.get(MODEL_NAME, MODEL_NAME)),
          "recall@100_zero_shot": ZERO_SHOT, "recall@100_tuned": RECALL_TUNED,
          "mode": "smoke" if SMOKE else "dry_run" if DRY else "full",
          "device": GPU, "amp": AMP.name, "commit": COMMIT,
          "emb_config": E.as_dict(), "ranker_config_for_samples": R.as_dict(),
          "seed": CFG.seed, "versions": library_versions()})
(EMB_DIR / "history.json").write_text(history.to_json(orient="records"))

print(json.dumps({k: MANIFEST[k] for k in ("model_name", "mode", "n_items", "n_queries", "dim", "md5")},
                 ensure_ascii=False, indent=1))
size_gb = sum(f.stat().st_size for f in EMB_DIR.rglob("*") if f.is_file()) / 2 ** 30
print(f"\nартефакт: {EMB_DIR} ({size_gb:.2f} ГБ) | всего времени: {(time.perf_counter() - START) / 60:.0f} мин")